In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 7.7 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='icosahedron31', repo_name='IEEE-CIS-Fraud-Detection', mlflow=True)



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=966638cc-74f5-40af-8749-e96065f33a23&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=3413b647b5972a1ddd10d8e40b567f41c6fb6488c326c08d9141be9c9016bb6b




Output()

Accessing as icosahedron31

Initialized MLflow to track repo "icosahedron31/IEEE-CIS-Fraud-Detection"

Repository icosahedron31/IEEE-CIS-Fraud-Detection initialized!

# Reading Data

In [4]:
df_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
df_identity.head(20)

,TransactionID,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987004,0.0,70787.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M
1,2987008,-5.0,98945.0,NaN,NaN,0.0,-5.0,NaN,NaN,NaN,...,mobile safari 11.0,32.0,1334x750,match_status:1,T,F,F,T,mobile,iOS Device
2,2987010,-5.0,191631.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
3,2987011,-5.0,221832.0,NaN,NaN,0.0,-6.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,NaN
4,2987016,0.0,7460.0,0.0,0.0,1.0,0.0,NaN,NaN,0.0,...,chrome 62.0,24.0,1280x800,match_status:2,T,F,T,T,desktop,MacOS
5,2987017,-5.0,61141.0,3.0,0.0,3.0,0.0,NaN,NaN,3.0,...,chrome 62.0,24.0,1366x768,match_status:2,T,F,T,T,desktop,Windows
6,2987022,-15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2987038,0.0,31964.0,0.0,0.0,0.0,-10.0,NaN,NaN,0.0,...,chrome 62.0,32.0,1920x1080,match_status:2,T,F,T,T,mobile,NaN
8,2987040,-10.0,116098.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows
9,2987048,-5.0,257037.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN,...,chrome 62.0,NaN,NaN,NaN,F,F,T,T,desktop,Windows


In [5]:
df_identity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144233 entries, 0 to 144232
Data columns (total 41 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionID  144233 non-null  int64  
 1   id_01          144233 non-null  float64
 2   id_02          140872 non-null  float64
 3   id_03          66324 non-null   float64
 4   id_04          66324 non-null   float64
 5   id_05          136865 non-null  float64
 6   id_06          136865 non-null  float64
 7   id_07          5155 non-null    float64
 8   id_08          5155 non-null    float64
 9   id_09          74926 non-null   float64
 10  id_10          74926 non-null   float64
 11  id_11          140978 non-null  float64
 12  id_12          144233 non-null  object 
 13  id_13          127320 non-null  float64
 14  id_14          80044 non-null   float64
 15  id_15          140985 non-null  object 
 16  id_16          129340 non-null  object 
 17  id_17          139369 non-nul

In [6]:
df_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")

In [7]:
df_transaction.info(), df_transaction.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB


(None,
        TransactionID        isFraud  TransactionDT  TransactionAmt  \
 count   5.905400e+05  590540.000000   5.905400e+05   590540.000000   
 mean    3.282270e+06       0.034990   7.372311e+06      135.027176   
 std     1.704744e+05       0.183755   4.617224e+06      239.162522   
 min     2.987000e+06       0.000000   8.640000e+04        0.251000   
 25%     3.134635e+06       0.000000   3.027058e+06       43.321000   
 50%     3.282270e+06       0.000000   7.306528e+06       68.769000   
 75%     3.429904e+06       0.000000   1.124662e+07      125.000000   
 max     3.577539e+06       1.000000   1.581113e+07    31937.391000   
 
                card1          card2          card3          card5  \
 count  590540.000000  581607.000000  588975.000000  586281.000000   
 mean     9898.734658     362.555488     153.194925     199.278897   
 std      4901.170153     157.793246      11.336444      41.244453   
 min      1000.000000     100.000000     100.000000     100.000000   
 2

In [8]:
df_transaction.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
df_train = df_transaction.merge(df_identity, on='TransactionID', how='left')
df_train.columns

Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5',
       ...
       'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38',
       'DeviceType', 'DeviceInfo'],
      dtype='object', length=434)

In [10]:
df_train.head(20)
df_train['isFraud']

0         0
1         0
2         0
3         0
4         0
         ..
590535    0
590536    0
590537    0
590538    0
590539    0
Name: isFraud, Length: 590540, dtype: int64

In [11]:
X = df_train.drop(columns=['isFraud'])
y = df_train['isFraud']
X.shape, y.shape

((590540, 433), (590540,))

# Train/test Split


In [12]:
from sklearn.model_selection import train_test_split



X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train.shape, X_val.shape

((472432, 433), (118108, 433))

# Baseline

In [39]:
cat_cols = X_train.select_dtypes(include='object').columns

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

In [40]:
from xgboost import XGBClassifier

model = XGBClassifier(
    device="cuda",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',   # important for speed
    random_state=42,
    enable_categorical=True,
    n_jobs=-1
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [27]:
import mlflow
import mlflow.xgboost
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay
)


def run_experiment(
    model,
    run_name,
    experiment_name,
    X_train,
    X_val,
    y_train,
    y_val
):
    # Set experiment
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_name=run_name):

        # Train
        model.fit(X_train, y_train)

        # Predict
        probs = model.predict_proba(X_val)[:, 1]
        preds = model.predict(X_val)
        probs_train = model.predict_proba(X_train)[:, 1]
        train_auc = roc_auc_score(y_train, probs_train)
        # Metrics
        auc = roc_auc_score(y_val, probs)
        accuracy = accuracy_score(y_val, preds)
        recall = recall_score(y_val, preds)
        precision = precision_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)

        fraud_mask = (y_val == 1)
        nonfraud_mask = (y_val == 0)

        mlflow.log_metrics({
            "train auc": train_auc,
            "auc": auc,
            "accuracy": accuracy,
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "fraud_mean_prob": probs[fraud_mask].mean(),
            "nonfraud_mean_prob": probs[nonfraud_mask].mean()
        })

        # Log model parameters if available
        if hasattr(model, "get_params"):
            params = {k: str(v) for k, v in model.get_params().items()}
            mlflow.log_params(params)


        
        fpr, tpr, _ = roc_curve(y_val, probs)

        plt.figure()
        plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve")
        plt.legend()

        mlflow.log_figure(plt.gcf(), "roc_curve.png")
        plt.close()

        precision_vals, recall_vals, _ = precision_recall_curve(y_val, probs)

        plt.figure()
        plt.plot(recall_vals, precision_vals)
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title("Precision-Recall Curve")

        mlflow.log_figure(plt.gcf(), "pr_curve.png")
        plt.close()

      
        cm = confusion_matrix(y_val, preds)

        plt.figure()
        ConfusionMatrixDisplay(cm).plot()
        plt.title("Confusion Matrix")

        mlflow.log_figure(plt.gcf(), "confusion_matrix.png")
        plt.close()

       
        plt.figure()

        plt.hist(
            probs[nonfraud_mask],
            bins=50,
            alpha=0.5,
            label="non_fraud"
        )

        plt.hist(
            probs[fraud_mask],
            bins=50,
            alpha=0.5,
            label="fraud"
        )

        plt.xlabel("Predicted Probability")
        plt.ylabel("Count")
        plt.title("Prediction Score Distribution")
        plt.legend()

        mlflow.log_figure(
            plt.gcf(),
            "score_distribution.png"
        )
        plt.close()

        # Log model artifact
        mlflow.sklearn.log_model(model, "model")

        return {
            "auc": auc,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }

# Preprocessing

In [14]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class MissingValueFilter(BaseEstimator, TransformerMixin):
    """
    Drops columns based on missing ratio observed during fit.
    Strict version: raises error if expected columns are missing at transform time.
    """

    def __init__(self, column_threshold=0.9, row_threshold=None):
        self.column_threshold = column_threshold
        self.row_threshold = row_threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()

        missing_ratio = X.isnull().mean()

        self.columns_to_keep_ = missing_ratio[
            missing_ratio <= self.column_threshold
        ].index.tolist()

        # store for strict validation
        self._fitted_columns_ = set(X.columns)

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        # strict schema validation (this prevents silent bugs)
        missing_cols = set(self.columns_to_keep_) - set(X.columns)

        if missing_cols:
            raise ValueError(
                f"Missing columns at transform time: {sorted(missing_cols)}"
            )

        # keep only valid columns
        X = X[self.columns_to_keep_]

        if self.row_threshold is not None:
            row_missing_ratio = X.isnull().mean(axis=1)
            X = X[row_missing_ratio <= self.row_threshold]

        return X

In [15]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin


class NAPreprocessor(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        numeric_strategy="median",
        categorical_strategy="mode"
    ):
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy

    def fit(self, X, y=None):
        X = X.copy()

        self.fill_values_ = {}

        for col in X.columns:

            # Numeric columns
            if pd.api.types.is_numeric_dtype(X[col]):

                if self.numeric_strategy == "mean":
                    value = X[col].mean()

                elif self.numeric_strategy == "median":
                    value = X[col].median()

             

            # Categorical columns
            else:

                if self.categorical_strategy == "mode":

                    mode_vals = X[col].mode()

                    if len(mode_vals) > 0:
                        value = mode_vals.iloc[0]
                    else:
                        value = "missing"

                elif self.categorical_strategy == "constant":
                    value = "missing"


            self.fill_values_[col] = value

        return self

    def transform(self, X):
        X = X.copy()

        for col, fill_value in self.fill_values_.items():

            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X

In [18]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cat_cols = None
        self.num_cols = None
        self.encoder = None
        self.medians = None

    def fit(self, X, y=None):
        X = X.copy()
        self.cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
        self.num_cols = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
        self.medians = X[self.num_cols].median()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            self.encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            self.encoder.fit(X[self.cat_cols])
        return self

    def transform(self, X):
        X = X.copy()
        if self.cat_cols:
            X[self.cat_cols] = X[self.cat_cols].fillna("missing")
            if self.encoder is not None:
                X[self.cat_cols] = self.encoder.transform(X[self.cat_cols])
        for col in self.num_cols:
            if col in X.columns:
                X[col] = X[col].fillna(self.medians[col])
        for col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")
        X = X.fillna(0)
        return X

In [19]:
import pandas as pd
import numpy as np

class WOEEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None, smoothing=0.5):
        self.cols = cols
        self.smoothing = smoothing
        self.woe_maps = {}

    def fit(self, X, y):
        X = X.copy()
        cols = self.cols or X.select_dtypes(include=["object", "category"]).columns.tolist()
        
        total_events = y.sum()
        total_non_events = (1 - y).sum()

        for col in cols:
            tmp = pd.DataFrame({"col": X[col], "target": y.values})
            stats = tmp.groupby("col")["target"].agg(["sum", "count"])
            stats.columns = ["events", "count"]
            stats["non_events"] = stats["count"] - stats["events"]

            # smoothing to avoid log(0)
            stats["dist_events"]     = (stats["events"] + self.smoothing) / (total_events + self.smoothing)
            stats["dist_non_events"] = (stats["non_events"] + self.smoothing) / (total_non_events + self.smoothing)
            stats["woe"] = np.log(stats["dist_events"] / stats["dist_non_events"])

            self.woe_maps[col] = stats["woe"].to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        for col, woe_map in self.woe_maps.items():
            if col in X.columns:
                X[col] = X[col].map(woe_map).fillna(0)  # unseen → 0 (neutral)
        return X

In [47]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            # Evaluated lazily at fit time — sees columns AFTER MissingValueFilter runs
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
)

pipeline = Pipeline([
    ("missing_filter", MissingValueFilter(column_threshold=0.97)),
    ("na", NAPreprocessor(
        numeric_strategy="median",
        categorical_strategy="mode"
    )),
    ("encoder", encoder),
    ("model", XGBClassifier(
        device="cuda",
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
        
    ))
])

In [48]:
run_experiment(pipeline, "XGBoost_cleaning", "XGBoost", X_train, X_val, y_train, y_val)

2026/05/04 09:24:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:24:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_cleaning at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/9d09f99ccbe84c3688a85f6ea933f2db
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


{'auc': np.float64(0.945698126907371),
 'accuracy': 0.9806278998882378,
 'precision': 0.9369409660107334,
 'recall': 0.49387081565299384,
 'f1': 0.6468045693115159}

<Figure size 640x480 with 0 Axes>

# Handling Imbalanced 

scale-pos weight

In [20]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print(scale_pos_weight)

27.769989647402717


In [23]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            # Evaluated lazily at fit time — sees columns AFTER MissingValueFilter runs
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
)

pipeline = Pipeline([
    
    ("encoder", encoder),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
        
    ))
])

In [37]:
woe_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
X_train = X_train.astype({c: "object" for c in X_train.select_dtypes("category").columns})
X_val  = X_val.astype({c: "object" for c in X_val.select_dtypes("category").columns})
pipeline = Pipeline([
    ("missing", MissingValueFilter()),
    ("woe", WOEEncoder()),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
        
    ))
])

In [39]:
params_to_try = [
    {"n_estimators": 1000, "learning_rate": 0.05,  "max_depth": 12, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 10, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 12, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1200, "learning_rate": 0.01, "max_depth": 10, "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 12, "subsample": 1.0, "colsample_bytree": 1.0}
]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_deep_{params}_featureEng", "XGBoost", X_train_eng, X_val_eng, y_train, y_val)
#run_experiment(pipeline, "XGBoost_scale_positives_and_woe", "XGBoost", X_train, X_val, y_train, y_val)

2026/05/04 19:42:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 19:42:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_deep_{'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 12, 'subsample': 0.8, 'colsample_bytree': 0.8}_featureEng at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/f883684d66ea493ba20e70975ba4bcf7
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 19:44:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 19:44:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_deep_{'n_estimators': 1000, 'learning_rate': 0.1, 'max_depth': 10, 'subsample': 1, 'colsample_bytree': 0.8}_featureEng at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/5620d65b831e49a0ad4bc8825028e23f
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 19:47:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 19:47:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

smote

In [53]:
import pandas as pd
from imblearn.pipeline import Pipeline  # ← imblearn, not sklearn
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OrdinalEncoder
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
)

pipeline = Pipeline([
    ("preprocess", FraudPreprocessor()),
    ("encode", encoder),
    ("smote", SMOTE(random_state=42)),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
    ))
])

In [54]:
params_to_try = [
    {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": 12, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 10, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 500, "learning_rate": 0.03, "max_depth": 5, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 800, "learning_rate": 0.01, "max_depth": 6, "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 6, "subsample": 1.0, "colsample_bytree": 1.0}
]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_{params}", "XGBoost", X_train, X_val, y_train, y_val)

2026/05/04 09:29:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:29:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/ff8e9b182f6744a4b483191ccdcb3be6
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:31:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:31:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/0b91fd8c8e0545edaba0c1d7498fec24
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:33:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:33:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 500, 'learning_rate': 0.01, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/11841f6169124d21a786864079f060c1
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

Undersampling

In [55]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from xgboost import XGBClassifier
encoder = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            # Evaluated lazily at fit time — sees columns AFTER MissingValueFilter runs
            make_column_selector(dtype_include=["object", "category"])
        )
    ],
    remainder="passthrough"
)


pipeline = Pipeline([
    
   
   
    ("encoder", encoder), 
    ("under", RandomUnderSampler(sampling_strategy=0.2, random_state=42)),
    ("model", XGBClassifier(
        device="cuda",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        scale_pos_weight=1,
        tree_method="hist",
        
    ))
])

In [56]:
params_to_try = [
    {"n_estimators": 200, "learning_rate": 0.1, "max_depth": 4},
    {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6},
    {"n_estimators": 500, "learning_rate": 0.01, "max_depth": 6},
]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_{params}", "XGBoost", X_train, X_val, y_train, y_val)

2026/05/04 09:34:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:34:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/e1127d86cbbd46779a97ed51cd2fca6e
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:34:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:34:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/2dab9541c8b946c29a8f14265b33c0bb
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 09:35:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:35:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 500, 'learning_rate': 0.01, 'max_depth': 6} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/0f6717bb255f4b4ca41975775043f58c
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

# Feature Engineering 

In [32]:
import pandas as pd
import numpy as np

def engineer_features(X, y=None):
    X = X.copy()

    X['amt_log'] = np.log1p(X['TransactionAmt'])
    X['amt_rounded'] = (X['TransactionAmt'] % 1 == 0).astype(int)  # whole number flag
    

    X['hour'] = (X['TransactionDT'] // 3600) % 24
    X['day_of_week'] = (X['TransactionDT'] // (3600 * 24)) % 7
    X['is_weekend'] = X['day_of_week'].isin([5, 6]).astype(int)

    X['email_match'] = (X['P_emaildomain'] == X['R_emaildomain']).astype(int)
    
    # --- Aggregates per card ---
    for col in ['card1', 'card2', 'card4', 'card6']:
        if col in X.columns:
            X[f'{col}_txn_count'] = X.groupby(col)['TransactionAmt'].transform('count')
            X[f'{col}_amt_mean']  = X.groupby(col)['TransactionAmt'].transform('mean')
            X[f'{col}_amt_std']   = X.groupby(col)['TransactionAmt'].transform('std')
            X[f'{col}_amt_dev']   = X['TransactionAmt'] - X[f'{col}_amt_mean']
            X[f'{col}_amt_zscore'] = X[f'{col}_amt_dev'] / (X[f'{col}_amt_std'] + 1e-9)

    for col in ['P_emaildomain', 'R_emaildomain']:
        if col in X.columns:
            X[f'{col}_freq'] = X.groupby(col)['TransactionAmt'].transform('count')

    if 'DeviceInfo' in X.columns:
        X['device_freq'] = X.groupby('DeviceInfo')['TransactionAmt'].transform('count')

    if y is not None:
        tmp = X.copy()
        tmp['target'] = y.values
        for col in ['card4', 'card6', 'P_emaildomain', 'DeviceType']:
            if col in X.columns:
                fraud_rate = tmp.groupby(col)['target'].transform('mean')
                X[f'{col}_fraud_rate'] = fraud_rate

    # --- Time since last transaction per card ---
    X = X.sort_values('TransactionDT')
    X['time_since_last_txn'] = X.groupby('card1')['TransactionDT'].diff().fillna(0)

    return X


# Usage
X_engineered = engineer_features(X, y)

In [33]:
from sklearn.model_selection import train_test_split



X_train_eng, X_val_eng, y_train, y_val = train_test_split(
    X_engineered, y,
    test_size=0.2,
    random_state=42
)

X_train_eng.shape, X_val_eng.shape


((472432, 467), (118108, 467))

In [24]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.email_freqs_ = {}
        self.fraud_rates_ = {}

    def fit(self, X, y=None):
        X = X.copy()

        for col in ['P_emaildomain', 'R_emaildomain']:
            if col in X.columns:
                self.email_freqs_[col] = (
                    X[col]
                    .value_counts()
                    .to_dict()
                )

        if y is not None:
            tmp = X.copy()
            tmp['_target'] = y.values

            for col in ['P_emaildomain', 'R_emaildomain']:
                if col in X.columns:
                    self.fraud_rates_[col] = (
                        tmp.groupby(col)['_target']
                        .mean()
                        .to_dict()
                    )

        return self


    def transform(self, X):
        X = X.copy()
        original_index = X.index

        # Amount features
        X['amt_log'] = np.log1p(X['TransactionAmt'])
        X['amt_rounded'] = (
            X['TransactionAmt'] % 1 == 0
        ).astype(int)

        # Time features
        X['hour'] = (
            X['TransactionDT'] // 3600
        ) % 24

        X['day_of_week'] = (
            X['TransactionDT'] // (3600 * 24)
        ) % 7

        X['is_weekend'] = (
            X['day_of_week'].isin([5, 6])
        ).astype(int)

        # Email features
        if {'P_emaildomain', 'R_emaildomain'}.issubset(X.columns):
            X['email_match'] = (
                X['P_emaildomain']
                == X['R_emaildomain']
            ).astype(int)

        for col, freq_map in self.email_freqs_.items():
            X[f'{col}_freq'] = X[col].map(freq_map)

        # Target encoding
        for col, fraud_map in self.fraud_rates_.items():
            X[f'{col}_fraud_rate'] = X[col].map(fraud_map)

        # Time since previous transaction
        if {'card1', 'TransactionDT'}.issubset(X.columns):

            tmp = X[['card1', 'TransactionDT']].copy()
            tmp = tmp.sort_values('TransactionDT')

            tmp['time_since_last_txn'] = (
                tmp.groupby('card1')['TransactionDT']
                .diff()
            )

            X['time_since_last_txn'] = (
                tmp['time_since_last_txn']
                .reindex(original_index)
            )

        return X

In [34]:
woe_cols = X_train_eng.select_dtypes(include=["object", "category"]).columns.tolist()
X_train_eng = X_train_eng.astype({c: "object" for c in X_train_eng.select_dtypes("category").columns})
X_val_eng  = X_val_eng.astype({c: "object" for c in X_val_eng.select_dtypes("category").columns})
pipeline = Pipeline([
    ("missingFilter", MissingValueFilter()), 

    ("woe", WOEEncoder()),
    ("fraud", FraudPreprocess())
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
        
    ))
])

<>:8: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:8: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_104/1084864579.py:8: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ("fraud", FraudPreprocess())


NameError: name 'FraudPreprocess' is not defined

In [73]:
params_to_try = [
    {"n_estimators": 1000, "learning_rate": 0.05,  "max_depth": 12, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 10, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 12, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1200, "learning_rate": 0.01, "max_depth": 10, "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 12, "subsample": 1.0, "colsample_bytree": 1.0}
]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_deep_{params}_cleaning", "XGBoost", X_train, X_val, y_train, y_val)
#run_experiment(pipeline, "XGBoost_scale_positives_and_woe", "XGBoost", X_train, X_val, y_train, y_val)

2026/05/04 10:38:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:38:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/2b730d2ae01e4521af8c7e3b0430628a
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 10:40:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:40:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_{'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 4, 'subsample': 0.8, 'colsample_bytree': 0.8} at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/900bd8340e8946b4aee8f5de6763a925
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


2026/05/04 10:41:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:42:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>

# Dropping some features

In [25]:
import numpy as np
import pandas as pd
import shap

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


class SHAPFeatureSelector(BaseEstimator, TransformerMixin):
    
    def __init__(
        self,
        sample_size=5000,
        drop_percent=0.2,
        random_state=42
    ):
        self.sample_size = sample_size
        self.drop_percent = drop_percent
        self.random_state = random_state
        
    def fit(self, X, y):
        
        # sample for speed
        n = min(self.sample_size, len(X))
        
        rng = np.random.RandomState(self.random_state)
        idx = rng.choice(len(X), n, replace=False)

        X_sample = X.iloc[idx]
        y_sample = y.iloc[idx] if hasattr(y, "iloc") else y[idx]

        # internal xgboost model for feature ranking
        model = XGBClassifier(
            device="cuda",
            tree_method="hist",
            eval_metric="auc",
            n_estimators=300,
            max_depth=4,
            learning_rate=0.1,
            random_state=self.random_state
        )

        model.fit(X_sample, y_sample)

        # SHAP
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_sample)

        importance = pd.Series(
            np.abs(shap_values).mean(axis=0),
            index=X.columns
        ).sort_values(ascending=False)

        threshold = importance.quantile(self.drop_percent)

        self.selected_columns_ = importance[
            importance > threshold
        ].index.tolist()

        print(
            f"Selected {len(self.selected_columns_)} "
            f"out of {X.shape[1]} features"
        )

        return self

    def transform(self, X):
        return X[self.selected_columns_]



In [47]:
pipeline = Pipeline([
    ("missing", MissingValueFilter()),
    ("woe", WOEEncoder()),
    ("fraud", FraudPreprocessor()),
    ("shap_select", SHAPFeatureSelector(
        sample_size=5000,
        drop_percent=0.2
    )),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
    ))
])

params_to_try = [
    {"n_estimators": 1000, "learning_rate": 0.05,  "max_depth": 15, "subsample": 0.8, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.1, "max_depth": 10, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1000, "learning_rate": 0.05, "max_depth": 12, "subsample": 1, "colsample_bytree": 0.8},
    {"n_estimators": 1200, "learning_rate": 0.01, "max_depth": 10, "subsample": 0.9, "colsample_bytree": 0.9},
    {"n_estimators": 800, "learning_rate": 0.05, "max_depth": 12, "subsample": 1.0, "colsample_bytree": 1.0}
]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_deep_{params}_featureSelection", "XGBoost", X_train, X_val, y_train, y_val)

Selected 206 out of 421 features


2026/05/04 20:14:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 20:15:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGBoost_deep_{'n_estimators': 1000, 'learning_rate': 0.05, 'max_depth': 15, 'subsample': 0.8, 'colsample_bytree': 0.8}_featureSelection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/5511566444dc416ab79ab790970fb856
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7
🏃 View run XGBoost_deep_{'n_estimators': 1000, 'learning_rate': 0.1, 'max_depth': 10, 'subsample': 1, 'colsample_bytree': 0.8}_featureSelection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/600b4dfcda22485a9b7ccd47db6cbb4c
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


KeyboardInterrupt: 

<Figure size 640x480 with 0 Axes>

In [28]:
pipeline = Pipeline([
    ("engineer", FeatureEngineer()),
    ("missing", MissingValueFilter()),
    ("woe", WOEEncoder()),
    ("fraud", FraudPreprocessor()),
    ("shap_select", SHAPFeatureSelector(
        sample_size=5000,
        drop_percent=0.2
    )),
    ("model", XGBClassifier(
        device="cuda",
        scale_pos_weight=scale_pos_weight,
        eval_metric="auc",
        tree_method="hist",
        random_state=42
    ))
])

params_to_try = [
    {"n_estimators": 1200, "learning_rate": 0.05,  "max_depth": 15, "subsample": 0.8, "colsample_bytree": 0.8},

]

for params in params_to_try:
    pipeline.set_params(**{f"model__{k}": v for k, v in params.items()})
    run_experiment(pipeline, f"XGBoost_deep_{params}_featureSelection", "XGBoost", X_train, X_val, y_train, y_val)

Selected 201 out of 432 features


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [09:54:59] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
2026/05/05 09:55:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 09:55:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more informatio

🏃 View run XGBoost_deep_{'n_estimators': 1200, 'learning_rate': 0.05, 'max_depth': 15, 'subsample': 0.8, 'colsample_bytree': 0.8}_featureSelection at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7/runs/22b546df12aa4d3fa175b895a1aced4c
🧪 View experiment at: https://dagshub.com/icosahedron31/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/7


<Figure size 640x480 with 0 Axes>